# Parquet Export Preview
Inspect PASSAGE Parquet shards under `outputs/export`, decode tensor/image bytes, and visualize a sample for a chosen resolution and split. Set `RESOLUTION`, `SPLIT`, and `ROW_INDEX` below, then step through the cells to inspect tensors and PNGs.

In [1]:
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq

try:
    from PIL import Image
except ImportError:
    Image = None

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
export_root = root / "outputs" / "export"

# Configuration
RESOLUTION = 256
SPLIT = "train"  # train | validation | test | calibration
ROW_INDEX = 0  # row index within the chosen shard


def decode_tensor(raw, resolution=None):
    if raw is None:
        return None
    if isinstance(raw, (bytes, bytearray, memoryview)):
        with BytesIO(raw) as bio:
            tensor = np.load(bio, allow_pickle=False)
    else:
        tensor = np.asarray(raw, dtype=np.float32)
    if resolution and tensor.ndim == 1:
        side = int(resolution)
        tensor = tensor.reshape((side, side, 3))
    if tensor.ndim != 3 or tensor.shape[2] != 3:
        raise ValueError(f"Expected tensor shape (H, W, 3), got {tensor.shape}")
    return tensor.astype(np.float32, copy=False)


def decode_image(raw, images_root=None, prefix=None):
    if raw is None:
        return None
    if hasattr(raw, "convert"):
        return np.array(raw.convert("RGB"))
    if isinstance(raw, (bytes, bytearray, memoryview)):
        if Image is None:
            raise ImportError("Install Pillow (`uv run pip install pillow`) to decode image bytes")
        with Image.open(BytesIO(raw)) as img:
            return np.array(img.convert("RGB"))
    if isinstance(raw, str):
        candidate = Path(raw)
        if images_root and not candidate.is_absolute():
            candidate = images_root / candidate
        if prefix and images_root and not candidate.is_absolute():
            candidate = images_root / prefix / raw
        if candidate.exists():
            if Image is None:
                raise ImportError("Install Pillow (`uv run pip install pillow`) to decode image paths")
            with Image.open(candidate) as img:
                return np.array(img.convert("RGB"))
    return None


def to_path_array(path_list):
    if not path_list:
        return None
    arr = np.asarray(path_list, dtype=int).reshape(-1, 2)
    return arr


def pick_shard(resolution, split):
    base = export_root / str(resolution) / split
    shards = sorted(base.glob("data-*.parquet"))
    return shards[0] if shards else None


shard = pick_shard(RESOLUTION, SPLIT)
tensor = None
image = None
path_free = None
path_obstacles = None
metadata = {}

if shard is None:
    print(f"No parquet shards found under {export_root}/{RESOLUTION}/{SPLIT}")
else:
    table = pq.read_table(shard)
    print(f"Loaded shard {shard.name} with {table.num_rows} rows")
    print(table.schema)
    row = table.slice(ROW_INDEX, 1).to_pylist()[0]
    metadata = row.get("metadata") or {}
    row_resolution = int(metadata.get("resolution") or RESOLUTION)
    tensor = decode_tensor(row.get("tensor"), resolution=row_resolution)
    path_free = to_path_array(row.get("path_free"))
    path_obstacles = to_path_array(row.get("path_obstacles"))
    images_root = shard.parent / "images"
    image = decode_image(row.get("image"), images_root=images_root, prefix=row.get("prefix"))
    print(
        f"row={ROW_INDEX}, resolution={row_resolution}, tensor shape={getattr(tensor, 'shape', None)}, "
        f"image={'yes' if image is not None else 'no'}"
    )

Loaded shard data-00000.parquet with 70 rows
id: string
image: binary
prefix: string
tensor: binary
path_free: list<element: list<element: int32>>
  child 0, element: list<element: int32>
      child 0, element: int32
path_obstacles: list<element: list<element: int32>>
  child 0, element: list<element: int32>
      child 0, element: int32
metadata: struct<calibration: struct<max: double, min: double>, crop: struct<bottom_right: struct<lat: double, (... 1544 chars omitted)
  child 0, calibration: struct<max: double, min: double>
      child 0, max: double
      child 1, min: double
  child 1, crop: struct<bottom_right: struct<lat: double, lon: double>, center: struct<lat: double, lon: double>, max (... 112 chars omitted)
      child 0, bottom_right: struct<lat: double, lon: double>
          child 0, lat: double
          child 1, lon: double
      child 1, center: struct<lat: double, lon: double>
          child 0, lat: double
          child 1, lon: double
      child 2, max_elevation

In [2]:
if tensor is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(tensor[:, :, 0], cmap="terrain")
    if path_free is not None and path_free.size:
        axes[0].plot(path_free[:, 1], path_free[:, 0], color="yellow", linewidth=1, label="free")
    if path_obstacles is not None and path_obstacles.size:
        axes[0].plot(path_obstacles[:, 1], path_obstacles[:, 0], color="green", linewidth=1, label="obstacles")
    axes[0].set_title("Tensor (channel 0)")
    axes[0].axis("off")
    if axes[0].lines:
        axes[0].legend(loc="upper right")

    if image is not None:
        axes[1].imshow(image)
        axes[1].set_title("PNG with paths")
        axes[1].axis("off")
    else:
        axes[1].set_visible(False)

    plt.tight_layout()
    plt.show()
elif shard is not None:
    print("Shard loaded but tensor missing?")
else:
    print("Nothing to show yet.")

In [3]:
# Optional: Hugging Face datasets demo (robust loading without explicit features)
import datasets as ds

parquet_ds = None
sample_ds = None
tensor_ds = None
image_ds = None
path_free_ds = None
path_obstacles_ds = None

if shard:
    parquet_ds = ds.load_dataset("parquet", data_files=str(shard), split="train")
    try:
        parquet_ds = parquet_ds.cast_column("image", ds.Image())
    except Exception:
        # If cast fails, continue with raw bytes (Pillow can still decode from bytes in notebook cells)
        pass
    sample_ds = parquet_ds[ROW_INDEX]
    print("keys:", list(sample_ds.keys()))
    tensor_ds = decode_tensor(
        sample_ds.get("tensor"), resolution=sample_ds.get("metadata", {}).get("resolution") or RESOLUTION
    )
    path_free_ds = to_path_array(sample_ds.get("path_free"))
    path_obstacles_ds = to_path_array(sample_ds.get("path_obstacles"))
    image_ds = decode_image(sample_ds.get("image"), images_root=shard.parent / "images", prefix=sample_ds.get("prefix"))
    print(f"tensor.shape={getattr(tensor_ds, 'shape', None)}, image={'yes' if image_ds is not None else 'no'}")
else:
    print("Set `shard` above before running this cell.")

redacted-other-home-02/projects/PASSAGE/passage/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


keys: ['id', 'image', 'prefix', 'tensor', 'path_free', 'path_obstacles', 'metadata']
tensor.shape=(256, 256, 3), image=yes


In [4]:
# Plot a sample from the huggingface `parquet_ds` (change ROW_INDEX to view other rows)
if sample_ds is not None and tensor_ds is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(tensor_ds[:, :, 0], cmap="terrain")
    if path_free_ds is not None and path_free_ds.size:
        axes[0].plot(path_free_ds[:, 1], path_free_ds[:, 0], color="yellow", linewidth=1, label="free")
    if path_obstacles_ds is not None and path_obstacles_ds.size:
        axes[0].plot(path_obstacles_ds[:, 1], path_obstacles_ds[:, 0], color="green", linewidth=1, label="obstacles")
    axes[0].set_title("Tensor (channel 0)")
    axes[0].axis("off")
    if axes[0].lines:
        axes[0].legend(loc="upper right")

    if image_ds is not None:
        axes[1].imshow(image_ds)
        axes[1].set_title("PNG with paths (datasets)")
        axes[1].axis("off")
    else:
        axes[1].set_visible(False)

    plt.tight_layout()
    plt.show()
elif shard is not None:
    print("Run the datasets loading cell above to populate `sample_ds`.")
else:
    print("Nothing to show yet.")